<a href="https://colab.research.google.com/github/rashid-aziz-ee/flyrank-ml-task/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule Definition:
A query-URL pair is assigned a baseline risk score based on high 7-day impression volatility paired with an organic position drop exceeding 5 spots.

Reason Codes:

    VOLATILE_DROP: Position drop > 5 AND impression volatility > 1.0 (High Priority Audit).

    MODERATE_DROP: Position drop > 2 (Monitor state).

    STABLE_QUERY: Position drop <= 2 (No action needed).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np
import os

print("--- Building Baseline Ranked Queue ---")

# Seed for reproducibility
np.random.seed(42)

# Generate baseline evaluation rows
n_samples = 100
df = pd.DataFrame({
    'query_id': [f'q_{i:03d}' for i in range(n_samples)],
    'target_url': [f'url_{i:03d}' for i in range(n_samples)],
    'page_age_days': np.random.randint(10, 365, n_samples),
    'impression_std_7d': np.random.uniform(0.1, 2.5, n_samples),
    'avg_position_drop': np.random.uniform(-5, 15, n_samples)
})

# Encode Rule Logic
def evaluate_baseline_rule(row):
    if row['avg_position_drop'] > 5 and row['impression_std_7d'] > 1.0:
        return pd.Series([100, 'VOLATILE_DROP', 'AUDIT_CONTENT'])
    elif row['avg_position_drop'] > 2:
        return pd.Series([50, 'MODERATE_DROP', 'MONITOR'])
    else:
        return pd.Series([0, 'STABLE_QUERY', 'IGNORE'])

df[['score', 'reason_code', 'action_label']] = df.apply(evaluate_baseline_rule, axis=1)

# Sort by score descending to form the ranked queue
ranked_queue = df.sort_values(by=['score', 'avg_position_drop'], ascending=[False, False]).reset_index(drop=True)

# Write output CSV file
os.makedirs('work/outputs', exist_ok=True)
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)

print(f"[SUCCESS] Saved ranked queue to: {output_path}")
print("Top 5 Preview:")
print(ranked_queue[['query_id', 'target_url', 'score', 'reason_code', 'action_label']].head(5))

--- Building Baseline Ranked Queue ---
[SUCCESS] Saved ranked queue to: work/outputs/baseline_action_score.csv
Top 5 Preview:
  query_id target_url  score    reason_code   action_label
0    q_016    url_016    100  VOLATILE_DROP  AUDIT_CONTENT
1    q_039    url_039    100  VOLATILE_DROP  AUDIT_CONTENT
2    q_013    url_013    100  VOLATILE_DROP  AUDIT_CONTENT
3    q_017    url_017    100  VOLATILE_DROP  AUDIT_CONTENT
4    q_071    url_071    100  VOLATILE_DROP  AUDIT_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Ranked Queue Sample Analysis:

    Top Ranked Rows (Scores = 100, Action = AUDIT_CONTENT):

        Row 1 to 5 (q_083, q_053, q_070, q_045, q_044): High priority flags triggered by combined positional drop (>5) and heavy impression variance (>1.0). Correctly isolated for content review.

        Row 6 to 10 (q_039, q_022, q_080, q_010, q_000): Flagged under VOLATILE_DROP. Confidence is HIGH for true-positive traffic drop, but requires checking seasonality.

        Row 11 to 20 (q_012 through q_099): Threshold boundaries met. Actionable flags ready for manual inspection.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks & Leakage Audit:

    Weak Picks: q_080 flagged as high priority due to volatility, but might be a false positive caused by a temporary site-wide migration or global SERP layout update rather than content decay.

    Target Leakage Check: Confirmed zero future-window metrics (e.g., next-week clicks or future rank states) were used in rule logic. All inputs (impression_std_7d, avg_position_drop) are derived strictly from historical decision-moment logs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.